# Stage 1: Non-Instruction Domain Fine-Tuning

This notebook demonstrates how to perform non-instruction (pre-training style) fine-tuning on raw IT Helpdesk domain knowledge text. We use the **Unsloth** library to accelerate training and reduce VRAM usage on an open-source model (**Qwen2.5-1.5B**).

### Step 1: Install Required Libraries
We install the Unsloth library, along with xformers, trl, and standard Hugging Face components.

In [ ]:
# Install Unsloth and standard ML libraries
# Uncomment the lines below to install if running in a GPU cloud env (like Colab)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers" "trl" peft accelerate bitsandbytes

### Step 2: Load Model and Tokenizer
We load `unsloth/Qwen2.5-1.5B` in 4-bit precision to save VRAM.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # None for auto-detection. Float16 for Tesla T4, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### Step 3: Configure LoRA / PEFT
We target attention projection matrices and feed-forward networks for parameter-efficient training.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank of adapter matrices
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16, # Scaling factor
    lora_dropout = 0, # Dropout is optimized at 0 for speed
    bias = "none",    # "none", "lora_only", or "all"
    use_gradient_checkpointing = "unsloth", # Saves VRAM by checkpointing
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### Step 4: Load and Prepare Raw Domain Text
We read `data/non_instruction_data.txt`, clean it, chunk it, and format it for next-token prediction pre-training.

In [ ]:
import os
from datasets import Dataset

# Check path compatibility (Colab vs Local)
data_path = "data/non_instruction_data.txt" if os.path.exists("data/non_instruction_data.txt") else "../data/non_instruction_data.txt"

# Read file paragraphs
with open(data_path, "r", encoding="utf-8") as f:
    text_content = f.read()

# Split into paragraphs, remove empty ones
paragraphs = [p.strip() for p in text_content.split("\n\n") if p.strip()]

# Convert to a Hugging Face Dataset
dataset = Dataset.from_dict({"text": paragraphs})
print(f"Loaded {len(dataset)} paragraphs of raw domain data.")

### Step 5: Configure and Run SFTTrainer
We configure standard pre-training style training. Since there is no Q&A structure, the target column is simply `text` and no prompt template is applied.

In [ ]:
import inspect
from trl import SFTTrainer
from transformers import TrainingArguments

# Try to import SFTConfig for newer TRL versions
try:
    from trl import SFTConfig
except ImportError:
    SFTConfig = None

if SFTConfig is not None:
    # Newer TRL config style (v0.8.0+)
    sig_config = inspect.signature(SFTConfig.__init__)
    sft_args = {
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 4,
        "warmup_steps": 5,
        "max_steps": 30, # Small steps for demonstration
        "learning_rate": 2e-4,
        "fp16": not torch.cuda.is_bf16_supported(),
        "bf16": torch.cuda.is_bf16_supported(),
        "logging_steps": 1,
        "optim": "adamw_8bit",
        "weight_decay": 0.01,
        "lr_scheduler_type": "linear",
        "seed": 3407,
        "output_dir": "outputs",
        "dataset_text_field": "text",
        "packing": False,
    }
    if "max_seq_length" in sig_config.parameters:
        sft_args["max_seq_length"] = max_seq_length
    else:
        sft_args["max_length"] = max_seq_length
        
    trainer_args = SFTConfig(**sft_args)
else:
    # Older TRL style
    trainer_args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    )

# Prepare trainer arguments
sig_trainer = inspect.signature(SFTTrainer.__init__)
trainer_kwargs = {
    "model": model,
    "train_dataset": dataset,
    "dataset_num_proc": 2,
    "args": trainer_args,
}

if SFTConfig is None:
    # If SFTConfig is None, these must be passed directly to SFTTrainer
    trainer_kwargs["dataset_text_field"] = "text"
    trainer_kwargs["max_seq_length"] = max_seq_length
    trainer_kwargs["packing"] = False

if "processing_class" in sig_trainer.parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)
trainer_stats = trainer.train()

### Step 6: Test Inference Post-Training
We evaluate how the model completes sentences related to our corporate policies.

In [ ]:
FastLanguageModel.for_inference(model) # Enable native fast inference

prompt = "Password Policy: All employee passwords must"
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Step 7: Save Adapters
We save the fine-tuned LoRA weights and tokenizers locally.

In [ ]:
import os
# Check path compatibility (Colab/Root vs Local)
adapters_dir = "adapters" if os.path.exists("data") else "../adapters"
save_path = os.path.join(adapters_dir, "non_instruction_lora")

# Option A: Save locally
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Stage 1 LoRA adapters saved successfully to {save_path}!")

# Option B: Push to Hugging Face Hub (Uncomment to use)
# model.push_to_hub("techsivam/non_instruction_lora", private=True)
# tokenizer.push_to_hub("techsivam/non_instruction_lora", private=True)